# Run 1 — Baseline evaluation of the models you already have

Evaluates **v1 scratch** (`Akzhai/Thesis_Scratch_1`), **v1 fine-tuned** (`Akzhai/Thesis_Phase_1_lr5e6`) and the **pretrained Qwen2.5-0.5B** on the same unseen data: perplexity (FineWeb-Edu + WikiText-103), 4-gram repetition, next-word continuations, and the downstream tasks. Results are saved to `baseline_compare_results.json` (and, if enabled, to the small private repo `Akzhai/Thesis_Eval`, **not** to your model repos).

**Kaggle settings:** GPU T4 · Internet On · Secret `HF_TOKEN` · dataset `qwenlab_v3_modular` attached. Set `QUICK_TEST = True` first (≈5 min) to check everything runs, then `False`.

In [1]:
# ===================== 1. SETTINGS =====================
QUICK_TEST = False        # True: ~5-minute test.  False: full baseline evaluation

SCRATCH_REPO, SUBFOLDER = "Akzhai/Thesis_Scratch_1", "hf_model"
FINETUNED_REPO          = "Akzhai/Thesis_Phase_1_lr5e6"
PRETRAINED              = "Qwen/Qwen2.5-0.5B"
RESULTS_REPO            = "Akzhai/Thesis_Eval"        # small private repo for result files (your model repos are never written to)

N_PPL_WINDOWS = 200       # 512-token windows per text set for perplexity
N_SUMM, N_QA, N_CORR = 150, 200, 200
N_INSTR, N_FIM, FIM_SPAN = 50, 100, 16
GEN_BATCH, SEQ_LEN, SEED = 8, 512, 42
FINEWEB_SKIP_DOCS = 150_000   # documents skipped so the FineWeb text is unseen by the v1 runs
UPLOAD_RESULTS = True

if QUICK_TEST:
    N_PPL_WINDOWS, N_SUMM, N_QA, N_CORR, N_INSTR, N_FIM, GEN_BATCH, FINEWEB_SKIP_DOCS, UPLOAD_RESULTS = 16, 8, 16, 16, 8, 8, 4, 500, False

LABELS = {"scratch": "v1 scratch (random start)", "finetuned": "v1 fine-tuned (pretrained start)", "pretrained": "pretrained Qwen2.5-0.5B"}
SHORT  = {"scratch": "v1_scratch", "finetuned": "finetuned", "pretrained": "pretrained"}
print("Mode:", "QUICK_TEST" if QUICK_TEST else "FULL", "|", LABELS)

Mode: FULL | {'scratch': 'v1 scratch (random start)', 'finetuned': 'v1 fine-tuned (pretrained start)', 'pretrained': 'pretrained Qwen2.5-0.5B'}


In [2]:
# ===================== 2. INSTALL qwenlab (writable copy; /kaggle/input is read-only) =====================
import subprocess, shutil, sys, zipfile
from pathlib import Path

def _find(pattern):
    r = subprocess.run(["find", "/kaggle/input", "-iname", pattern], capture_output=True, text=True)
    return [p for p in r.stdout.strip().split("\n") if p]

def _pkg_roots(paths):
    return [Path(p).parent for p in paths if (Path(p).parent / "src" / "qwenlab").is_dir()]

roots = _pkg_roots(_find("pyproject.toml"))
if not roots:
    zips = _find("*.zip")
    assert zips, "No qwenlab dataset under /kaggle/input: attach qwenlab_v3_modular (Add Input)"
    unz = Path("/kaggle/working/qwenlab_unzipped"); unz.mkdir(exist_ok=True)
    for z in zips:
        zipfile.ZipFile(z).extractall(unz)
    roots = _pkg_roots([str(p) for p in unz.rglob("pyproject.toml")])
assert roots, "Found files but no folder with pyproject.toml next to src/qwenlab"

WORK_PKG = Path("/kaggle/working/qwenlab_src")
if WORK_PKG.exists():
    shutil.rmtree(WORK_PKG)
shutil.copytree(roots[0], WORK_PKG)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(WORK_PKG), "rouge-score"], check=True)
sys.path.insert(0, str(WORK_PKG / "src"))
print("qwenlab installed:", WORK_PKG)

qwenlab installed: /kaggle/working/qwenlab_src


In [3]:
# ===================== 3. LOAD TOKENIZER AND THE 3 MODELS =====================
import json, math, time, re, string, difflib, subprocess, sys, gc
from collections import Counter
import numpy as np, torch, torch.nn.functional as F
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from qwenlab.tasks.prompt_response import render_chatml, ChatExample, DEFAULT_INSTRUCTIONS
from qwenlab.tasks.fim import fim_arrays
from qwenlab.data.synthetic import split_sentences, make_correction_pairs
from qwenlab.evaluation.repetition import repeated_ngram_fraction

assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU in the notebook settings"
login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
device = "cuda"
AC = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

def load_hf(repo, sub=None):
    """Loads a model in fp32 (works with both old and new transformers argument names)."""
    kw = dict(subfolder=sub) if sub else {}
    try:
        m = AutoModelForCausalLM.from_pretrained(repo, dtype=torch.float32, **kw)
    except TypeError:
        m = AutoModelForCausalLM.from_pretrained(repo, torch_dtype=torch.float32, **kw)
    return m.to(device).eval()

tok = AutoTokenizer.from_pretrained(PRETRAINED)
for repo, sub in ((SCRATCH_REPO, SUBFOLDER), (FINETUNED_REPO, "hf_model")):
    assert tok.get_vocab() == AutoTokenizer.from_pretrained(repo, subfolder=sub).get_vocab(), f"Tokenizer mismatch in {repo}"
tok.padding_side = "left"
EOS = tok.eos_token_id
PAD = tok.pad_token_id if tok.pad_token_id is not None else EOS
IM_START, IM_END = "<|im_start|>", "<|im_end|>"
IM_END_ID = tok.convert_tokens_to_ids(IM_END)
FIM_IDS = [tok.convert_tokens_to_ids(t) for t in ("<|fim_prefix|>", "<|fim_middle|>", "<|fim_suffix|>")]

models = {"scratch": load_hf(SCRATCH_REPO, SUBFOLDER), "finetuned": load_hf(FINETUNED_REPO, "hf_model"), "pretrained": load_hf(PRETRAINED)}
V = models["pretrained"].config.vocab_size
print("3 models loaded | vocab:", V, "| GPU memory:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

try:
    from rouge_score import rouge_scorer
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge-score"], check=True)
    from rouge_score import rouge_scorer

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

3 models loaded | vocab: 151936 | GPU memory: 5.9 GB


In [4]:
# ===================== 4. HELPER FUNCTIONS =====================
HELPER_INSTR = "You are a helpful assistant."

def enc_len(s): return len(tok.encode(s))
def fits(e):
    p, r = render_chatml(e, IM_START, IM_END)
    return enc_len(p) + enc_len(r) <= SEQ_LEN

def to_pairs(examples):
    return [(tok.encode(p), tok.encode(r)) for p, r in (render_chatml(e, IM_START, IM_END) for e in examples)]

def build_sft_arrays(chunk, pad_id):
    seqs = [p + r for p, r in chunk]
    L = max(len(s) for s in seqs)
    ids = [s + [pad_id] * (L - len(s)) for s in seqs]
    att = [[1] * len(s) + [0] * (L - len(s)) for s in seqs]
    lab = [[-100] * len(p) + list(r) + [-100] * (L - len(p) - len(r)) for p, r in chunk]
    return ids, att, lab

def nll_windows(model, windows, bs=4):
    total, count = 0.0, 0
    for i in range(0, len(windows), bs):
        batch = torch.tensor([list(map(int, w)) for w in windows[i:i + bs]], device=device)
        x, y = batch[:, :-1], batch[:, 1:]
        with torch.no_grad(), torch.autocast("cuda", dtype=AC):
            logits = model(input_ids=x).logits
        total += F.cross_entropy(logits.float().reshape(-1, V), y.reshape(-1), reduction="sum").item()
        count += y.numel()
    return total / count

def sft_loss(model, pairs, bs=4):
    total, count = 0.0, 0
    for i in range(0, len(pairs), bs):
        ids, att, lab = build_sft_arrays(pairs[i:i + bs], PAD)
        ids, att, lab = (torch.tensor(a, device=device) for a in (ids, att, lab))
        with torch.no_grad(), torch.autocast("cuda", dtype=AC):
            logits = model(input_ids=ids, attention_mask=att).logits
        total += F.cross_entropy(logits[:, :-1].float().reshape(-1, V), lab[:, 1:].reshape(-1), ignore_index=-100, reduction="sum").item()
        count += int((lab[:, 1:] != -100).sum().item())
    return total / count

def masked_loss(model, xs, ys, ms, bs=4):
    total, count = 0.0, 0
    for i in range(0, len(xs), bs):
        x = torch.tensor(xs[i:i + bs], device=device); y = torch.tensor(ys[i:i + bs], device=device)
        m = torch.tensor(ms[i:i + bs], device=device, dtype=torch.bool)
        with torch.no_grad(), torch.autocast("cuda", dtype=AC):
            logits = model(input_ids=x).logits
        ce = F.cross_entropy(logits.float().reshape(-1, V), y.reshape(-1), reduction="none").reshape(y.shape)
        total += ce[m].sum().item(); count += int(m.sum().item())
    return total / count

def generate_texts(model, prompts, max_new_tokens, bs=None):
    bs = bs or GEN_BATCH
    outs = []
    for i in range(0, len(prompts), bs):
        enc = tok(prompts[i:i + bs], return_tensors="pt", padding=True).to(device)
        with torch.no_grad(), torch.autocast("cuda", dtype=AC):
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=PAD, eos_token_id=[IM_END_ID, EOS])
        outs += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=False)
    return outs

def generate_from_ids(model, id_lists, max_new_tokens, bs=None):
    bs = bs or GEN_BATCH
    outs = []
    for i in range(0, len(id_lists), bs):
        chunk = id_lists[i:i + bs]; L = max(len(s) for s in chunk)
        ids = torch.full((len(chunk), L), PAD, dtype=torch.long); att = torch.zeros((len(chunk), L), dtype=torch.long)
        for j, s in enumerate(chunk):
            ids[j, L - len(s):] = torch.tensor(s); att[j, L - len(s):] = 1
        with torch.no_grad(), torch.autocast("cuda", dtype=AC):
            out = model.generate(input_ids=ids.to(device), attention_mask=att.to(device), max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=PAD, eos_token_id=[IM_END_ID, EOS])
        outs += [row[L:].tolist() for row in out]
    return outs

def clean_generation(text, fmt="chat", one_line=False, max_sentences=None):
    """fmt: 'chat' | 'plain' (cut at the first blank line) | 'raw' (only remove stop tokens)."""
    for stop in ("<|im_end|>", "<|endoftext|>", "<|im_start|>"):
        text = text.split(stop)[0]
    text = text.strip()
    if fmt == "plain" and not one_line:
        text = text.split("\n\n")[0].split("Article:")[0].strip()
    if one_line:
        text = text.split("\n")[0].strip()
    if max_sentences:
        parts = re.split(r"(?<=[.!?\u0964\u0965\u06d4\u061f])\s+", text)
        text = " ".join(parts[:max_sentences]).strip()
    return text

def rouge_all(preds, refs):
    sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    scores = [sc.score(r, p) for p, r in zip(preds, refs)]
    return {k: 100 * float(np.mean([s[k].fmeasure for s in scores])) for k in ("rouge1", "rouge2", "rougeL")}

_PUNCT = set(string.punctuation)
def _norm(s):
    s = "".join(ch for ch in s.lower() if ch not in _PUNCT)
    return " ".join(re.sub(r"\b(a|an|the)\b", " ", s).split())
def squad_f1(pred, refs):
    best = 0.0
    for r in refs:
        p_t, r_t = _norm(pred).split(), _norm(r).split()
        common = sum((Counter(p_t) & Counter(r_t)).values())
        if common == 0 or not p_t or not r_t: continue
        p, rc = common / len(p_t), common / len(r_t)
        best = max(best, 2 * p * rc / (p + rc))
    return best

def sim(a, b): return difflib.SequenceMatcher(None, a, b).ratio()

def fim_prompt(window, span=None, n_pre=100, n_suf=100):
    span = span or FIM_SPAN
    w = [int(t) for t in window]
    pre, mid, suf = w[:n_pre], w[n_pre:n_pre + span], w[n_pre + span:n_pre + span + n_suf]
    return [FIM_IDS[0]] + pre + [FIM_IDS[2]] + suf + [FIM_IDS[1]], mid

M, HIGHER = {}, {}
def put(metric, model_key, value, higher):
    M.setdefault(metric, {})[model_key] = float(value); HIGHER[metric] = higher

In [5]:
# ===================== 5. LOAD THE UNSEEN TEST DATA =====================
t0 = time.time()
wt = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="test")
text = "\n".join(t for t in wt["text"] if t.strip())[: N_PPL_WINDOWS * SEQ_LEN * 6]
ids = tok(text)["input_ids"]
wiki_windows = [ids[i * SEQ_LEN: i * SEQ_LEN + SEQ_LEN + 1] for i in range(min(N_PPL_WINDOWS, (len(ids) - 1) // SEQ_LEN))]
print(f"WikiText-103 test: {len(wiki_windows)} windows")

fw_windows = None
try:
    stream = load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT", split="train", streaming=True).skip(FINEWEB_SKIP_DOCS)
    need, buf, batch = N_PPL_WINDOWS * SEQ_LEN + 1, [], []
    for ex in stream:
        if ex["text"] and ex["text"].strip(): batch.append(ex["text"])
        if len(batch) >= 64:
            for d in tok(batch)["input_ids"]: buf.extend(d + [EOS])
            batch = []
            if len(buf) >= need: break
    fw_windows = [buf[i * SEQ_LEN: i * SEQ_LEN + SEQ_LEN + 1] for i in range(min(N_PPL_WINDOWS, (len(buf) - 1) // SEQ_LEN))]
    print(f"FineWeb-Edu (unseen documents): {len(fw_windows)} windows")
except Exception as e:
    print("FineWeb-Edu unavailable:", repr(e)[:150])
fim_windows = (fw_windows or wiki_windows)[:max(N_FIM, 16)]

def shorten(article, max_tokens=400):
    a = tok.encode(article)
    return tok.decode(a[:max_tokens]) if len(a) > max_tokens else article
summ = []
for ex in load_dataset("abisee/cnn_dailymail", "3.0.0", split="test", streaming=True):
    summ.append((shorten(ex["article"]), ex["highlights"]))
    if len(summ) >= N_SUMM: break

instr = []
for ex in load_dataset("tatsu-lab/alpaca", split="train").shuffle(seed=SEED):
    user = ex["instruction"] + (("\n\n" + ex["input"]) if ex["input"] else "")
    e = ChatExample(HELPER_INSTR, user, ex["output"])
    if fits(e): instr.append(e)
    if len(instr) >= N_INSTR: break

qa = []
for ex in load_dataset("rajpurkar/squad", split="validation").shuffle(seed=SEED):
    e = ChatExample(DEFAULT_INSTRUCTIONS["qa"], f"Context: {ex['context']}\n\nQuestion: {ex['question']}", ex["answers"]["text"][0])
    if fits(e): qa.append((e, ex["context"], ex["question"], list(ex["answers"]["text"])))
    if len(qa) >= N_QA: break

ds_v = load_dataset("rajpurkar/squad", split="validation").shuffle(seed=SEED + 7)
val_ctx = list(dict.fromkeys(ds_v[i]["context"] for i in range(min(len(ds_v), 400))))
val_sents = split_sentences(" ".join(val_ctx))
corr = []
for bad, clean in make_correction_pairs(val_sents, N_CORR * 3, seed=SEED + 100):
    e = ChatExample(DEFAULT_INSTRUCTIONS["correction"], bad, clean)
    if fits(e): corr.append((e, bad, clean))
    if len(corr) >= N_CORR: break
print(f"Data ready in {time.time()-t0:.0f}s | summaries={len(summ)}, instructions={len(instr)}, qa={len(qa)}, correction={len(corr)}")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes:           |  0.00B            

wikitext-103-raw-v1/train-00000-of-00002(…): reconstructing file:   0%|          |  0.00B /  157MB            

wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:           |  0.00B            

...

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Data ready in 23s | summaries=150, instructions=50, qa=200, correction=200


In [6]:
# ===================== 6. PERPLEXITY AND HELD-OUT LOSS =====================
summ_pairs = to_pairs([ChatExample(DEFAULT_INSTRUCTIONS["summarize"], a, s) for a, s in summ])
instr_pairs, qa_pairs, corr_pairs = to_pairs(instr), to_pairs([q[0] for q in qa]), to_pairs([c[0] for c in corr])
fim_x, fim_y, fim_m = fim_arrays(fim_windows, SEQ_LEN, *FIM_IDS, seed=SEED)

t0 = time.time()
for name, m in models.items():
    w_loss = nll_windows(m, wiki_windows)
    put("ppl/WikiText-103 loss", name, w_loss, False)
    put("ppl/WikiText-103 perplexity", name, math.exp(w_loss), False)
    if fw_windows:
        f_loss = nll_windows(m, fw_windows)
        put("ppl/FineWeb-Edu loss (unseen)", name, f_loss, False)
        put("ppl/FineWeb-Edu perplexity (unseen)", name, math.exp(f_loss), False)
    put("loss/summarize", name, sft_loss(m, summ_pairs), False)
    put("loss/instruct", name, sft_loss(m, instr_pairs), False)
    put("loss/qa", name, sft_loss(m, qa_pairs), False)
    put("loss/correction", name, sft_loss(m, corr_pairs), False)
    put("loss/fim (missing span)", name, masked_loss(m, fim_x, fim_y, fim_m), False)
    print(f"{name:10s} loss & perplexity done ({time.time()-t0:.0f}s)")

def show_table(prefix, title):
    keys = [k for k in M if k.startswith(prefix)]
    print(f"\n{title:62s}" + "".join(f"{SHORT[n]:>14s}" for n in models))
    for k in keys:
        vals = M[k]; best = (max if HIGHER[k] else min)(vals, key=vals.get)
        print(f"{k.split('/',1)[1][:61]:62s}" + "".join(f"{vals.get(n, float('nan')):13.3f}{'*' if n == best else ' '}" for n in models))

show_table("ppl/", "PERPLEXITY AND LM LOSS (lower is better, * = best)")
show_table("loss/", "TASK ANSWER LOSS (lower is better, * = best)")

scratch    loss & perplexity done (47s)
finetuned  loss & perplexity done (98s)
pretrained loss & perplexity done (148s)

PERPLEXITY AND LM LOSS (lower is better, * = best)                v1_scratch     finetuned    pretrained
WikiText-103 loss                                                     4.931         2.934         2.824*
WikiText-103 perplexity                                             138.543        18.794        16.846*
FineWeb-Edu loss (unseen)                                             4.269         2.753*        2.805 
FineWeb-Edu perplexity (unseen)                                      71.478        15.690*       16.522 

TASK ANSWER LOSS (lower is better, * = best)                      v1_scratch     finetuned    pretrained
summarize                                                             3.760         2.020*        3.235 
instruct                                                              3.576         1.291*        1.721 
qa                                   

In [7]:
# ===================== 7. GENERATION QUALITY AND REPETITION =====================
GEN = {}
t0 = time.time()

# next-word continuations + repetition (120 new tokens with a short prompt = still inside the 512-token window)
CONT_PROMPTS = ["The capital of France is", "In 1969, the first humans landed on", "The three primary colors are",
                "Water boils at a temperature of", "In science, the most important idea is",
                "Once upon a time, in a small village near the river,"]
for name, m in models.items():
    raw_120 = generate_texts(m, CONT_PROMPTS, max_new_tokens=120)
    long_texts = [clean_generation(t, "raw") for t in raw_120]
    guarded = [clean_generation(t, "raw", max_sentences=2) for t in raw_120]
    GEN[("continuation_120_tokens", name)] = long_texts
    GEN[("continuation_2_sentences", name)] = guarded
    put("repetition/4-gram repeat % (120 new tokens)", name, 100 * np.mean([repeated_ngram_fraction(tok.encode(t), 4) for t in long_texts]), False)
    put("repetition/4-gram repeat % (cut at 2 sentences)", name, 100 * np.mean([repeated_ngram_fraction(tok.encode(t)[:48], 4) for t in guarded]), False)

summ_refs = [s for _, s in summ]
summ_prompts = {"chat":  [render_chatml(ChatExample(DEFAULT_INSTRUCTIONS["summarize"], a, ""), IM_START, IM_END)[0] for a, _ in summ],
                "plain": [f"Article:\n{a}\n\nSummary:\n" for a, _ in summ]}
for fmt in ("chat", "plain"):
    for name, m in models.items():
        preds = [clean_generation(t, fmt) for t in generate_texts(m, summ_prompts[fmt], 100)]
        GEN[("summarize", name, fmt)] = preds
        put(f"summarize/ROUGE-L ({fmt} prompt)", name, rouge_all(preds, summ_refs)["rougeL"], True)

qa_prompts = {"chat":  [render_chatml(ChatExample(DEFAULT_INSTRUCTIONS["qa"], f"Context: {c}\n\nQuestion: {q}", ""), IM_START, IM_END)[0] for _, c, q, _ in qa],
              "plain": [f"Context: {c}\n\nQuestion: {q}\nShort answer:" for _, c, q, _ in qa]}
for fmt in ("chat", "plain"):
    for name, m in models.items():
        preds = [clean_generation(t, fmt, one_line=True) for t in generate_texts(m, qa_prompts[fmt], 24)]
        GEN[("qa", name, fmt)] = preds
        put(f"qa/F1 % ({fmt} prompt)", name, 100 * np.mean([squad_f1(p, a) for p, (_, _, _, a) in zip(preds, qa)]), True)

corr_prompts = {"chat":  [render_chatml(ChatExample(DEFAULT_INSTRUCTIONS["correction"], bad, ""), IM_START, IM_END)[0] for _, bad, _ in corr],
                "plain": [f"Fix the spelling and grammar of the sentence below.\n\nSentence: {bad}\nCorrected sentence:" for _, bad, _ in corr]}
for fmt in ("chat", "plain"):
    for name, m in models.items():
        preds = [clean_generation(t, fmt, one_line=True) for t in generate_texts(m, corr_prompts[fmt], 64)]
        GEN[("correction", name, fmt)] = preds
        put(f"correction/similarity to clean % ({fmt} prompt)", name, 100 * np.mean([sim(p, c) for p, (_, _, c) in zip(preds, corr)]), True)

instr_prompts = [render_chatml(ChatExample(e.instruction, e.user_content, ""), IM_START, IM_END)[0] for e in instr]
for name, m in models.items():
    preds = [clean_generation(t, "chat") for t in generate_texts(m, instr_prompts, 120)]
    GEN[("instruct", name, "chat")] = preds
    put("instruct/ROUGE-L vs reference (chat prompt)", name, rouge_all(preds, [e.response for e in instr])["rougeL"], True)

fim_cases = [fim_prompt(w) for w in fim_windows[:N_FIM]]
for name, m in models.items():
    outs = generate_from_ids(m, [p for p, _ in fim_cases], FIM_SPAN)
    accs = [np.mean([a == b for a, b in zip(o[:len(mid)] + [-1] * max(0, len(mid) - len(o)), mid)]) for o, (_, mid) in zip(outs, fim_cases)]
    put(f"fim/span token accuracy % ({FIM_SPAN}-token gap)", name, 100 * np.mean(accs), True)

for prefix, title in (("repetition/", "REPETITION RATE (lower is better)"), ("summarize/", "SUMMARIZATION"), ("qa/", "QUESTION ANSWERING"),
                      ("correction/", "SPELLING CORRECTION"), ("instruct/", "INSTRUCTIONS"), ("fim/", "FILL-IN-THE-MIDDLE")):
    show_table(prefix, title)
print(f"\nGeneration finished in {time.time()-t0:.0f}s")


REPETITION RATE (lower is better)                                 v1_scratch     finetuned    pretrained
4-gram repeat % (120 new tokens)                                     82.305        52.672        17.929*
4-gram repeat % (cut at 2 sentences)                                 22.848         0.000*        0.741 

SUMMARIZATION                                                     v1_scratch     finetuned    pretrained
ROUGE-L (chat prompt)                                                10.650        22.276*       19.842 
ROUGE-L (plain prompt)                                                6.364        19.484        20.525*

QUESTION ANSWERING                                                v1_scratch     finetuned    pretrained
F1 % (chat prompt)                                                   19.890        43.570*       10.054 
F1 % (plain prompt)                                                   4.074        47.702        55.037*

SPELLING CORRECTION                                

In [8]:
# ===================== 8. SAVE (local file, then the small results repo) =====================
payload = {"metrics": M, "higher_is_better": HIGHER, "labels": LABELS,
           "samples": {str(k): v[:4] for k, v in GEN.items()}, "quick_test": QUICK_TEST}
json.dump(payload, open("/kaggle/working/baseline_compare_results.json", "w"), indent=2)
print("Saved /kaggle/working/baseline_compare_results.json  (download it from the Output tab)")
if UPLOAD_RESULTS:
    try:
        api.create_repo(RESULTS_REPO, private=True, exist_ok=True)
        api.upload_file(path_or_fileobj="/kaggle/working/baseline_compare_results.json", path_in_repo="baseline_compare_results.json",
                        repo_id=RESULTS_REPO, commit_message="Run 1: baseline evaluation before the v3 modular training")
        print("Also uploaded to", RESULTS_REPO)
    except Exception as e:
        print("Hub upload skipped (the local file is saved):", repr(e)[:150])

Saved /kaggle/working/baseline_compare_results.json  (download it from the Output tab)
Also uploaded to Akzhai/Thesis_Eval
